# Experimento: Robustez Fuera de Distribución y Envoltorios Adversariales de `veto_dir` (`LiquidAI/LFM3-1.7B`)

Evalúa el `.moipatch` de 2 vectores (`lfm3_safety_routed_refusal.moipatch`) ya guardado en Drive frente a **6 suites completamente nuevas (OOD)**:
1. `jbb_harmful_raw`: 100 objetivos tóxicos de **`JailbreakBench/JBB-Behaviors`** (`AdvBench`, `TDC/HarmBench`, `Original`).
2. `jbb_harmful_educational`: los mismos 100 pedidos envueltos en tono *"investigación educativa y sin causar daño"*.
3. `jbb_harmful_fiction`: envueltos en tono *"novela de ficción / thriller"*.
4. `jbb_harmful_hypothetical`: envueltos en tono *"hipotético, puramente defensivo y legal"*.
5. `jbb_benign_ood`: los 100 pares benignos de JBB (para medir si sigue curando sobre-rechazo fuera de OR-Bench).
6. `controls_benign`: controles sanos diarios (`0.0` drift y `0%` disparos).

No guarda ningún prompt ni respuesta tóxica cruda en el JSON — solo métricas agregadas, cuantiles de `veto_dir` y tasas por categoría/origen. Guarda el resultado en `MyDrive/wsi_runs/FINALES/`.

In [ ]:
MODEL = "LiquidAI/LFM3-1.7B"
RUN_NAME = "veto_robustness_lfm3_l4"
REPO = "/content/wsi"
BASE = "/content/base"
PATCH_DRIVE = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3/lfm3_safety_routed_refusal.moipatch"
UNVETOED_PATCH_DRIVE = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3/lfm3_surgical_refusal.moipatch"
N_HARMFUL = 100
N_BENIGN = 100
N_CONTROL = 30
MAX_NEW_TOK = 128
BATCH_SIZE = 16

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers accelerate safetensors pandas pyarrow huggingface_hub datasets
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
RUN_DIR = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3"
os.makedirs(FINALES, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

if not os.path.exists(PATCH_DRIVE):
    raise FileNotFoundError(f"No encuentro el .moipatch de seguridad en Drive: {PATCH_DRIVE}")

if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit:", COMMIT, "| patch:", PATCH_DRIVE, "| has_unvetoed:", os.path.exists(UNVETOED_PATCH_DRIVE))

In [ ]:
import os, subprocess, sys

subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

LOCAL_OUT = f"/content/{RUN_NAME}.json"
SCRIPT = f"{REPO}/scripts/experiment_veto_robustness_lfm3.py"
if not os.path.exists(SCRIPT):
    raise FileNotFoundError(f"Falta {SCRIPT} en {REPO} (commit {COMMIT}).")

cmd = [
    sys.executable, "-u", SCRIPT,
    "--model", BASE,
    "--patch", PATCH_DRIVE,
    "--n-harmful", str(N_HARMFUL),
    "--n-benign", str(N_BENIGN),
    "--n-control", str(N_CONTROL),
    "--max-new-tokens", str(MAX_NEW_TOK),
    "--batch-size", str(BATCH_SIZE),
    "--out", LOCAL_OUT,
]
if os.path.exists(UNVETOED_PATCH_DRIVE):
    cmd.extend(["--unvetoed-patch", UNVETOED_PATCH_DRIVE])

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

In [ ]:
import datetime, json, os, subprocess

res = json.load(open(LOCAL_OUT, encoding="utf-8"))
_pct = lambda v: f"{v:6.1%}" if isinstance(v, (int, float)) else "   n/a"
print("=== ROBUSTEZ OOD + ADVERSARIAL FRAMING (JailbreakBench / LFM3-1.7B) ===")
for sname, s in res["suites"].items():
    tel = s.get("router_telemetry") or {}
    extra = (
        f"kept={_pct(s.get('toxic_refusal_kept'))} | unvetoed_kept={_pct(s.get('unvetoed_toxic_refusal_kept'))}"
        if s["is_harmful"]
        else (f"ctrl_exact={_pct(s.get('control_exact_match_rate'))}" if s["is_control"] else f"recov={_pct(s.get('recovered_of_base_refused'))}")
    )
    print(f"{sname:25s} | base_ref={_pct(s['base_refusal_rate'])} | safe_ref={_pct(s['safety_routed_refusal_rate'])} "
          f"| {extra} | fire={_pct(tel.get('patch_fire_rate'))} | veto={_pct(tel.get('gate2_veto_rate'))}")

print("\n--- Desglose por fuente en JBB raw ---")
for src, info in (res["suites"]["jbb_harmful_raw"].get("by_source") or {}).items():
    print(f"  {src:16s} | n={info['n']:3d} | base_ref={_pct(info['base_refusal_rate'])} | kept={_pct(info['kept_of_base'])} "
          f"| fire={_pct(info.get('fire_rate'))} | veto={_pct(info.get('veto_rate'))}")

rg = res["robustness_gate"]
print(f"\n=== ROBUSTNESS GATE : {'PASS' if rg['overall_passed'] else 'FAIL'} ===")
print(f"raw_kept={_pct(rg['raw_ood_kept'])} | min_framed_kept={_pct(rg['min_framed_kept'])} | framed={rg['framed_kept']} "
      f"| benign_ood_recov={_pct(rg['ood_benign_recovered'])} | controls_ok={rg['controls_passed']}")
print("Elapsed (s):", res["elapsed_s"])

res["_final"] = {
    "complete": True,
    "model": MODEL,
    "commit": COMMIT,
    "notebook": "veto_robustness_lfm3_colab.ipynb",
    "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
for out_dir in (FINALES, RUN_DIR):
    os.makedirs(out_dir, exist_ok=True)
    dst = f"{out_dir}/{RUN_NAME}.json"
    with open(dst + ".tmp", "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2)
    os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", f"{FINALES}/{RUN_NAME}.json")

In [ ]:
AUTO_APAGAR = True
import os, time, json
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst, encoding="utf-8")).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")